In [109]:
import pandas as pd
import re
from nltk import ngrams
from gensim.models import Word2Vec

import sys

pd.options.display.max_rows = 9999

In [110]:
df = pd.read_csv("/kaggle/input/datasets/thngonquang/trienkieu/truyen_kieu_data.txt",sep="/", names=["row"], encoding="utf8").dropna()
df.head(10)

,row
0,"1..Trăm năm trong cõi người ta,"
1,2..Chữ tài chữ mệnh khéo là ghét nhau.
2,"3..Trải qua một cuộc bể dâu,"
3,4..Những điều trông thấy mà đau đớn lòng.
4,"5.. Lạ gì bỉ sắc tư phong,"
5,6..Trời xanh quen thói má hồng đánh ghen.
6,"7..Cảo thơm lần giở trước đèn,"
7,8..Phong tình có lục còn truyền sử xanh.
8,"9,,Rằng năm Gia Tĩnh triều Minh,"
9,"10.. Bốn phương phẳng lặng, hai kinh vững vàng."


In [111]:
def transform_row(row):
    # Xóa số dòng ở đầu câu
    row = re.sub(r"^[0-9\.]+", "", row)
    
    # Xóa dấu chấm, phẩy, hỏi ở cuối câu
    row = re.sub(r"[\.,\?]+$", "", row)
    
    # Xóa tất cả dấu chấm, phẩy, chấm phẩy, chấm thang, ... trong câu
    row = row.replace(",", " ").replace(".", " ") \
        .replace(";", " ").replace("“", " ") \
        .replace(":", " ").replace("”", " ") \
        .replace('"', " ").replace("'", " ") \
        .replace("!", " ").replace("?", " ")
    
    row = row.strip()
    return row 

df["row"] = df.row.apply(transform_row)
df.head(10)

,row
0,Trăm năm trong cõi người ta
1,Chữ tài chữ mệnh khéo là ghét nhau
2,Trải qua một cuộc bể dâu
3,Những điều trông thấy mà đau đớn lòng
4,Lạ gì bỉ sắc tư phong
5,Trời xanh quen thói má hồng đánh ghen
6,Cảo thơm lần giở trước đèn
7,Phong tình có lục còn truyền sử xanh
8,Rằng năm Gia Tĩnh triều Minh
9,Bốn phương phẳng lặng hai kinh vững vàng


In [112]:
!pip install unidecode

In [113]:
from unidecode import unidecode

df['row'] = df['row'].apply(unidecode)

In [114]:
def kieu_ngram(string, n=1):
    gram_str = list(ngrams(string.split(), n))
    return [ " ".join(gram).lower() for gram in gram_str ]

df["context"] = df.row.apply(lambda t: kieu_ngram(t, 1))

In [119]:
df.head()

,row,context
0,Tram nam trong coi nguoi ta,"[tram, nam, trong, coi, nguoi, ta]"
1,Chu tai chu menh kheo la ghet nhau,"[chu, tai, chu, menh, kheo, la, ghet, nhau]"
2,Trai qua mot cuoc be dau,"[trai, qua, mot, cuoc, be, dau]"
3,Nhung dieu trong thay ma dau don long,"[nhung, dieu, trong, thay, ma, dau, don, long]"
4,La gi bi sac tu phong,"[la, gi, bi, sac, tu, phong]"


In [115]:
train_data = df.context.tolist()
len(train_data)

3258

In [120]:
train_data

[['tram', 'nam', 'trong', 'coi', 'nguoi', 'ta'],
 ['chu', 'tai', 'chu', 'menh', 'kheo', 'la', 'ghet', 'nhau'],
 ['trai', 'qua', 'mot', 'cuoc', 'be', 'dau'],
 ['nhung', 'dieu', 'trong', 'thay', 'ma', 'dau', 'don', 'long'],
 ['la', 'gi', 'bi', 'sac', 'tu', 'phong'],
 ['troi', 'xanh', 'quen', 'thoi', 'ma', 'hong', 'danh', 'ghen'],
 ['cao', 'thom', 'lan', 'gio', 'truoc', 'den'],
 ['phong', 'tinh', 'co', 'luc', 'con', 'truyen', 'su', 'xanh'],
 ['rang', 'nam', 'gia', 'tinh', 'trieu', 'minh'],
 ['bon', 'phuong', 'phang', 'lang', 'hai', 'kinh', 'vung', 'vang'],
 ['co', 'nha', 'vien', 'ngoai', 'ho', 'vuong'],
 ['gia', 'tu', 'nghi', 'cung', 'thuong', 'thuong', 'buc', 'trung'],
 ['mot', 'trai', 'con', 'thu', 'rot', 'long'],
 ['vuong', 'quan', 'la', 'chu', 'noi', 'dong', 'nho', 'gia'],
 ['dau', 'long', 'hai', 'a', 'to', 'nga'],
 ['thuy', 'kieu', 'la', 'chi', 'em', 'la', 'thuy', 'van'],
 ['mai', 'cot', 'cach', 'tuyet', 'tinh', 'than'],
 ['mot', 'nguoi', 'mot', 've', 'muoi', 'phan', 'ven', 'muoi'],


In [116]:
model = Word2Vec(
    sentences=train_data,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

In [123]:
print("Vector cua 'nguoi':")
print(model.wv["meo"])

Vector cua 'nguoi':
[-0.03797093  0.03054873 -0.00915117  0.01236338  0.0055868  -0.05826037
  0.02115287  0.12559041 -0.0619019   0.01139682 -0.0346159  -0.08971232
 -0.0204097   0.0352181   0.01583916 -0.04772744  0.01508851 -0.06821186
 -0.02276903 -0.09649548  0.01541459  0.00640842  0.01750849 -0.02193315
 -0.04140026  0.00050186 -0.01859617 -0.07538374 -0.07083924 -0.01097911
  0.05920227  0.01210741  0.02729442 -0.03557362 -0.00946785  0.10011178
  0.00113263 -0.06628458 -0.01138064 -0.10017753  0.00909525 -0.05217248
 -0.03516078 -0.00344048  0.06783099  0.00854603 -0.0394425  -0.05173456
  0.05486289  0.01019658  0.01674911 -0.05923717  0.01196398  0.01500988
 -0.04558603  0.09709134  0.02085961 -0.01417848 -0.10519388  0.03950917
  0.00815846  0.03504018  0.00907398 -0.04811696 -0.07098563  0.02840621
  0.02409684  0.0544113  -0.04978818  0.07647236 -0.06163311  0.01102837
  0.05561261 -0.01785803  0.08186265  0.03714843  0.03912627 -0.02279056
 -0.04875237  0.04649788 -0.016

In [126]:
model.wv.most_similar("nguoi")

[('thay', 0.9984477162361145),
 ('tren', 0.9981642961502075),
 ('chuoc', 0.9981529116630554),
 ('ngat', 0.998148500919342),
 ('yen', 0.9979919195175171),
 ('han', 0.9979797005653381),
 ('khanh', 0.9979323744773865),
 ('tung', 0.9978998899459839),
 ('quen', 0.9978971481323242),
 ('tri', 0.9978575706481934)]